### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

In [ ]:
#@title 🚀 System Initialization & Asset Acquisition
import subprocess
import time
from IPython.display import display, HTML

def get_progress_html(task_desc, progress_pct, is_done=False):
    color = "#22c55e" if is_done else "#38bdf8"
    bar_color = "#22c55e" if is_done else "linear-gradient(90deg, #3b82f6, #10b981)"
    spinner = "none" if is_done else "inline-block"
    return f"""
    <div style="background:#09090b; border:1px solid #27272a; border-radius:12px; padding:15px; font-family:'JetBrains Mono', monospace; color:{color}; width:100%; max-width:800px; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
        <div style="display:flex; gap:6px; margin-bottom:10px; border-bottom:1px solid #27272a; padding-bottom:8px;">
            <div style="width:10px; height:10px; border-radius:50%; background:#ef4444;"></div>
            <div style="width:10px; height:10px; border-radius:50%; background:#eab308;"></div>
            <div style="width:10px; height:10px; border-radius:50%; background:#22c55e;"></div>
            <span style="margin-left:auto; color:#64748b; font-size:12px;">chucky-installer v1.0</span>
        </div>
        <div style="font-size:13px; line-height:1.6; margin-bottom: 12px; display: flex; align-items: center; gap: 8px;">
            <span style="display:{spinner}; width:14px; height:14px; border: 2px solid {color}; border-top-color: transparent; border-radius: 50%; animation: spin 1s linear infinite;"></span>
            >> {task_desc}
        </div>
        <div style="width: 100%; background: #1e293b; border-radius: 4px; overflow: hidden; height: 6px;">
            <div style="width: {progress_pct}%; background: {bar_color}; height: 100%; transition: width 0.4s ease;"></div>
        </div>
        <style>@keyframes spin {{ 100% {{ transform: rotate(360deg); }} }}</style>
    </div>
    """

handle = display(HTML(get_progress_html("Initializing Build Sequence...", 0)), display_id="installer")

tasks = [
    ("Installing core system packages...", "pip install -q uv gdown aria2 ipywidgets gradio websocket-client"),
    ("Upgrading Python package installer...", "uv pip install --system --upgrade pip -q"),
    ("Cloning ComfyUI base repository...", "git clone -q https://github.com/comfyanonymous/ComfyUI.git"),
    ("Installing Torch & backend dependencies...", "uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q && cd ComfyUI && uv pip install --system -r requirements.txt -q"),
    ("Cloning required custom node extensions...", "cd ComfyUI/custom_nodes && git clone -q https://github.com/rgthree/rgthree-comfy && git clone -q https://github.com/chflame163/ComfyUI_LayerStyle && git clone -q https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes && git clone -q https://github.com/city96/ComfyUI-GGUF"),
    ("Resolving external extension dependencies...", "uv pip install --system gguf blend_modes -q && uv pip install --system -r /content/ComfyUI/custom_nodes/ComfyUI_LayerStyle/requirements.txt -q && uv pip install --system -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt -q"),
    ("Downloading Flux 9B UNET weights...", "aria2c --console-log-level=error -q -c -x 16 -s 16 -k 1M https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q8_0.gguf -d /content/ComfyUI/models/unet -o flux-2-klein-9b-Q8_0.gguf"),
    ("Downloading Text Encoders & VAE...", "aria2c --console-log-level=error -q -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/flux2-klein-9B/resolve/main/split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors -d /content/ComfyUI/models/clip -o qwen_3_8b_fp8mixed.safetensors && aria2c --console-log-level=error -q -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors -d /content/ComfyUI/models/vae -o flux2-vae.safetensors"),
    ("Downloading Virtual Try-On LoRA...", "aria2c --console-log-level=error -q -c -x 16 -s 16 -k 1M https://huggingface.co/fal/flux-klein-9b-virtual-tryon-lora/resolve/main/flux-klein-tryon-comfy.safetensors -d /content/ComfyUI/models/loras -o flux-klein-tryon-comfy.safetensors"),
    ("Fetching customized Try-On Workflow JSON...", "wget -q https://github.com/AICHUCKY/Comfyui-Workflows/raw/AICHUCKY-patch-1/Flux%209b%20virtual%20try.json -O /content/ComfyUI/tryon_workflow.json")
]

for i, (desc, cmd) in enumerate(tasks):
    pct = int((i / len(tasks)) * 100)
    handle.update(HTML(get_progress_html(desc, pct)))
    subprocess.run(cmd, shell=True, capture_output=True)

handle.update(HTML(get_progress_html("Environment Setup Complete! Server Ready.", 100, is_done=True)))

In [ ]:
#@title 🎨 Launch Virtual Try-On Studio App
import os
import json
import sys
import time
import uuid
import socket
import threading
import subprocess
import urllib.request
import urllib.parse
import shutil
import warnings
import gradio as gr
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
import websocket

def get_progress_html(task_desc, progress_pct, is_done=False, is_error=False):
    if is_error:
        color = "#ef4444"
        bar_color = "#ef4444"
        spinner = "none"
    else:
        color = "#22c55e" if is_done else "#38bdf8"
        bar_color = "#22c55e" if is_done else "linear-gradient(90deg, #3b82f6, #10b981)"
        spinner = "none" if is_done else "inline-block"

    return f"""
    <div style="background:#09090b; border:1px solid #27272a; border-radius:12px; padding:15px; font-family:'JetBrains Mono', monospace; color:{color}; width:100%; max-width:800px; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
        <div style="display:flex; gap:6px; margin-bottom:10px; border-bottom:1px solid #27272a; padding-bottom:8px;">
            <div style="width:10px; height:10px; border-radius:50%; background:#ef4444;"></div>
            <div style="width:10px; height:10px; border-radius:50%; background:#eab308;"></div>
            <div style="width:10px; height:10px; border-radius:50%; background:#22c55e;"></div>
            <span style="margin-left:auto; color:#64748b; font-size:12px;">chucky-inference v1.0</span>
        </div>
        <div style="font-size:13px; line-height:1.6; margin-bottom: 12px; display: flex; align-items: center; gap: 8px;">
            <span style="display:{spinner}; width:14px; height:14px; border: 2px solid {color}; border-top-color: transparent; border-radius: 50%; animation: spin 1s linear infinite;"></span>
            >> {task_desc}
        </div>
        <div style="width: 100%; background: #1e293b; border-radius: 4px; overflow: hidden; height: 6px;">
            <div style="width: {progress_pct}%; background: {bar_color}; height: 100%; transition: width 0.4s ease;"></div>
        </div>
        <style>@keyframes spin {{ 100% {{ transform: rotate(360deg); }} }}</style>
    </div>
    """

init_handle = display(HTML(get_progress_html("Booting Headless Server & Preparing Web UI...", 40)), display_id="launcher")

workflow_json_path = "/content/ComfyUI/tryon_workflow.json"
comfy_input_dir = "/content/ComfyUI/input/"
comfy_output_dir = "/content/ComfyUI/output/"
os.makedirs(comfy_input_dir, exist_ok=True)
os.makedirs(comfy_output_dir, exist_ok=True)

process = subprocess.Popen(
    [sys.executable, "main.py", "--lowvram"],
    cwd="/content/ComfyUI",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

def is_server_ready(port, timeout=180):
    start_time = time.time()
    while time.time() - start_time < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                return True
        time.sleep(1.5)
    return False

if not is_server_ready(8188):
    init_handle.update(HTML(get_progress_html("SYSTEM ERROR: Backend failed to start. Restart the Colab session.", 100, is_error=True)))
    raise RuntimeError("Backend failed to start.")

init_handle.update(HTML(get_progress_html("Backend connected successfully! Generating your public URL below...", 100, is_done=True)))

def run_tryon_workflow(person_path, garment_path, prompt_text):
    if not person_path or not garment_path:
        yield None, None, get_progress_html("SYSTEM ERROR: Missing Target Person or Target Garment image.", 100, is_error=True)
        return

    yield None, None, get_progress_html("Initializing AI Synthesis Pipeline...", 10)

    p_filename = f"p_{uuid.uuid4().hex[:6]}.png"
    g_filename = f"g_{uuid.uuid4().hex[:6]}.png"
    shutil.copy(person_path, os.path.join(comfy_input_dir, p_filename))
    shutil.copy(garment_path, os.path.join(comfy_input_dir, g_filename))

    yield None, None, get_progress_html("Image assets staged and routed to backend tensors.", 20)

    with open(workflow_json_path, 'r', encoding='utf-8') as f:
        prompt_data = json.load(f)

    if "270" in prompt_data: prompt_data["270"]["inputs"]["image"] = p_filename
    if "278" in prompt_data: prompt_data["278"]["inputs"]["image"] = g_filename
    if "330" in prompt_data: prompt_data["330"]["inputs"]["text"] = prompt_text
    if "356" in prompt_data: prompt_data["356"]["inputs"]["unet_name"] = "flux-2-klein-9b-Q8_0.gguf"

    if "157" in prompt_data: prompt_data["157"]["inputs"]["image"] = ["271", 0]

    server_address = "127.0.0.1:8188"
    client_id = str(uuid.uuid4())

    payload = {"prompt": prompt_data, "client_id": client_id}
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(f"http://{server_address}/prompt", data=data)

    try:
        res = json.loads(urllib.request.urlopen(req).read())
        prompt_id = res['prompt_id']
    except Exception as e:
        yield None, None, get_progress_html(f"CONNECTION REFUSED: {e}", 100, is_error=True)
        return

    yield None, None, get_progress_html(f"Architecture Active. Task ID Dispatched: {prompt_id}", 30)

    ws = websocket.WebSocket()
    ws.connect(f"ws://{server_address}/ws?clientId={client_id}")

    while True:
        out = ws.recv()
        if isinstance(out, str):
            message = json.loads(out)
            if message['type'] == 'progress':
                data = message['data']
                progress_val = int(30 + ((data['value'] / data['max']) * 60))
                yield None, None, get_progress_html(f"Synthesizing Matrix: Node {data['node']} - Step {data['value']} / {data['max']}", progress_val)

            elif message['type'] == 'executing':
                data = message['data']
                if data['node'] is None and data['prompt_id'] == prompt_id:
                    yield None, None, get_progress_html("Synthesis Operations Concluded. Saving VAE Outputs...", 90)
                    break

    req = urllib.request.Request(f"http://{server_address}/history/{prompt_id}")
    history = json.loads(urllib.request.urlopen(req).read())[prompt_id]

    output_filepaths = []
    for node_id in history['outputs']:
        node_output = history['outputs'][node_id]
        if 'images' in node_output:
            for img in node_output['images']:
                url_values = urllib.parse.urlencode({"filename": img['filename'], "subfolder": img['subfolder'], "type": img['type']})
                req = urllib.request.Request(f"http://{server_address}/view?{url_values}")
                img_data = urllib.request.urlopen(req).read()
                output_filepath = os.path.join(comfy_output_dir, f"tryon_{uuid.uuid4().hex[:6]}_{img['filename']}")
                with open(output_filepath, "wb") as f:
                    f.write(img_data)
                output_filepaths.append(output_filepath)

    if len(output_filepaths) >= 2:
        img1, img2 = output_filepaths[0], output_filepaths[1]
    elif len(output_filepaths) == 1:
        img1, img2 = output_filepaths[0], None
    else:
        img1, img2 = None, None

    if img1 or img2:
        yield img1, img2, get_progress_html("RENDER SUCCESSFUL. Output(s) securely saved to /ComfyUI/output/.", 100, is_done=True)
    else:
        yield None, None, get_progress_html("FAILURE: No image payload returned from GPU.", 100, is_error=True)

custom_css = """
body, .gradio-container {
    background: #09090b !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    color: #e2e8f0 !important;
}
footer { display: none !important; }
.app-wrapper {
    max-width: 1400px !important;
    margin: 0 auto;
    padding: 2rem;
}
.studio-header {
    text-align: center;
    padding: 3rem 2rem;
    background: linear-gradient(135deg, #18181b 0%, #09090b 100%);
    border: 1px solid #27272a;
    border-radius: 20px;
    margin-bottom: 2rem;
    box-shadow: 0 20px 40px rgba(0,0,0,0.6);
    animation: fadeIn 0.8s ease-out;
}
.studio-header h1 {
    margin: 0;
    font-weight: 800;
    font-size: 2.5rem;
    background: linear-gradient(to right, #38bdf8, #818cf8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    letter-spacing: -0.5px;
}
.glass-card {
    background: rgba(24, 24, 27, 0.7) !important;
    backdrop-filter: blur(16px);
    border-radius: 16px !important;
    border: 1px solid rgba(255,255,255,0.05) !important;
    padding: 1.5rem !important;
    box-shadow: 0 10px 30px rgba(0,0,0,0.4) !important;
}
.card-title {
    font-size: 1.25rem;
    font-weight: 600;
    color: #f8fafc;
    margin-bottom: 1rem;
    padding-bottom: 0.5rem;
    border-bottom: 1px solid #27272a;
}
input, textarea {
    background: #09090b !important;
    border: 1px solid #27272a !important;
    border-radius: 8px !important;
    color: #f8fafc !important;
}
input:focus, textarea:focus {
    border-color: #38bdf8 !important;
    box-shadow: 0 0 0 1px #38bdf8 !important;
}
.action-btn {
    background: linear-gradient(135deg, #3b82f6, #6366f1) !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    padding: 1rem !important;
    font-size: 1.1rem !important;
    transition: all 0.3s ease !important;
    text-transform: uppercase;
    letter-spacing: 1px;
}
.action-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 10px 25px rgba(59, 130, 246, 0.4) !important;
}
@keyframes fadeIn {
    from { opacity: 0; transform: translateY(-10px); }
    to { opacity: 1; transform: translateY(0); }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Base()) as app:
    with gr.Column(elem_classes="app-wrapper"):
        gr.HTML("""
            <div class="studio-header">
                <h3 style="color: #ef4444; margin-bottom: 0.5rem; text-transform: uppercase; font-size: 0.9rem; letter-spacing: 1px;">🔴 Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #38bdf8; text-decoration: none;">AI With Chucky</a></h3>
                <h1>Virtual Try-On Studio</h1>
            </div>
        """)

        with gr.Row():
            with gr.Column(scale=1, elem_classes="glass-card"):
                gr.HTML('<div class="card-title">🛠️ Configuration Pipeline</div>')
                person_input = gr.Image(type="filepath", label="🧍 Target Person")
                garment_input = gr.Image(type="filepath", label="👕 Target Garment")
                prompt_input = gr.Textbox(
                    label="📝 Semantic Guidance (Prompt)",
                    value="A photorealistic virtual try-on, the subject is wearing the target garment. Preserve the original person's identity, body type, pose, and the original background perfectly, highly detailed, 8k.",
                    lines=3
                )
                execute_btn = gr.Button("Execute Generation", elem_classes="action-btn")

                gr.HTML('<div class="card-title" style="margin-top: 1.5rem;">🖥️ Execution Logs</div>')
                status_console = gr.HTML(get_progress_html("System Idle. Awaiting structural inputs...", 0))

            with gr.Column(scale=1.5, elem_classes="glass-card"):
                gr.HTML('<div class="card-title">✨ Final Output Renders</div>')
                with gr.Row():
                    out_img_1 = gr.Image(label="Render Component 1", interactive=False, show_label=True)
                    out_img_2 = gr.Image(label="Render Component 2", interactive=False, show_label=True)

    execute_btn.click(
        fn=run_tryon_workflow,
        inputs=[person_input, garment_input, prompt_input],
        outputs=[out_img_1, out_img_2, status_console]
    )

app.launch(share=True, inline=False, debug=False)